In [ ]:
# Precision audit Cell — paste after standalone Cells 1–5.
# Only reads existing config/ref; never assigns daily, replay, result or QC states.
from pathlib import Path
import hashlib, json, platform
import numpy as np
import pandas as pd
import xarray as xr
# Edit only these paths. Existing run is strictly read-only.
RUN_ROOT = Path('/data1/huangy/fig6/NC/v12/four_init_full_pipeline_NCVIconcurrent/ncvi_daily_standalone/run_001')
PRECISION_OUTPUT = RUN_ROOT.parent / 'ncvi_float32_audit_run_001'
AUDIT_REFERENCE_COMMIT = '705063b'  # source baseline, NOT the HPC execution HEAD
if PRECISION_OUTPUT.exists(): raise ValueError('Choose a new precision output directory')
run_manifest = json.loads((RUN_ROOT/'ncvi_daily_export_manifest.json').read_text())
config = run_manifest['config']
ref = pd.read_csv(config['stage_table'])
ref['window'] = ref.stage.replace({'Stage-I_dry':'Stage-I'})
ref = ref[ref.init_date.isin(['2023-06-08','2023-06-12']) & ref.window.isin(['Stage-I','Total'])]
def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()
def precision_load_si_pair(spec, init, hindcast=False):
    """Open explicitly pinned GRIB files only; validate level/units/member identity.

    Reforecast selector mirrors MLR: first exact init-date match, otherwise all
    records. The retained forecast-reference list must exactly match config.
    Nonmatching month/day or hour rejects the MLR fallback rather than using it.
    """
    import xarray as xr
    parts=[];audit=[]
    for role in ['cf','pf']:
        item=spec[role];path=Path(item['path'])
        if digest(path)!=item['sha256']:
            raise ValueError('GRIB source hash differs from pinned config')
        with xr.open_dataset(path,engine='cfgrib',backend_kwargs={'indexpath':'','filter_by_keys':{'shortName':'pv','typeOfLevel':'theta','level':320}}) as ds:
            if 'pv' not in ds:
                raise ValueError('Actual variable must be pv')
            a=ds.pv
            if a.attrs.get('GRIB_typeOfLevel')!='theta':
                raise ValueError('Require proven isentropic theta level')
            level=next((c for c in ['theta','potentialTemperature'] if c in a.coords),None)
            if level is None or np.size(a[level])!=1 or float(np.asarray(a[level]).ravel()[0])!=320:
                raise ValueError('Require exact 320 K; no nearest-layer substitution')
            if level in a.dims:a=a.squeeze(level,drop=True)
            units=str(a.attrs.get('units',''));normalized=units.lower().replace(' ','').replace('**','^')
            if normalized not in ['km^2kg^-1s^-1','k*m^2*kg^-1*s^-1','pvu']:
                raise ValueError(f'Unknown PV units: {units}; no magnitude heuristic')
            if normalized=='pvu': raise ValueError('Old MLR SI replay requires actual SI source; no reverse conversion')
            factor=1.
            if 'time' not in a.coords:raise ValueError('Missing forecast-reference time')
            refs=pd.DatetimeIndex(np.atleast_1d(a.time.values))
            matches=np.where(refs.normalize()==pd.Timestamp(init))[0]
            if not hindcast and len(matches)!=1:raise ValueError('Realtime init not unique')
            if matches.size:
                chosen=refs[[matches[0]]]
                if 'time' in a.dims:a=a.isel(time=int(matches[0]),drop=True)
            else:
                if not hindcast:raise ValueError('Missing realtime init')
                chosen=refs
            if [str(x) for x in chosen]!=item['selected_reference_times']:
                raise ValueError('Selected reference-time sample set differs from explicit config')
            if any((t.month,t.day,t.hour,t.minute)!=(pd.Timestamp(init).month,pd.Timestamp(init).day,0,0) for t in chosen):
                raise ValueError('Reforecast init-day/hour mismatch')
            if role=='cf':
                if 'number' in a.dims:
                    if a.sizes['number']!=1:raise ValueError('CF must have one member')
                    a=a.assign_coords(number=[0])
                else:a=a.expand_dims(number=[0])
            elif 'number' not in a.dims or not np.array_equal(a.number.values,np.arange(1,11 if hindcast else 51)):
                raise ValueError('PF IDs must be 1..10 for reforecast or 1..50 for realtime; no renumbering')
            lat=a.latitude;lon=a.longitude
            a=a.sel(latitude=lat[(lat>=35)&(lat<=50)],longitude=lon[(lon>=115)&(lon<=130)])
            if a.sizes['latitude']==0 or a.sizes['longitude']==0 or not np.isfinite(a.values).all():
                raise ValueError('Empty region/missing grid values; no silent skip')
            attrs=dict(a.attrs)
            regional=(a.mean(['latitude','longitude'],skipna=False)*factor).load()
            parts.append(a.load())
            audit.append(dict(init_date=init,role=('reforecast_' if hindcast else 'realtime_')+role,
                              path=str(path),sha256=item['sha256'],variable='pv',level=320,units=units,
                              pvu_multiplier=factor,reference_times=json.dumps([str(t) for t in chosen]),
                              sample_coordinates=json.dumps({d:[str(x) for x in regional[d].values] for d in regional.dims if d not in ['step','number']}),
                              attributes=json.dumps(attrs,default=str),spatial_mean='arithmetic grid-cell mean',
                              legacy_spatial_mean='cos(latitude) weighted',region='35-50N 115-130E'))
    merged=xr.concat(parts,dim='number',join='exact',compat='equals').sortby('number')
    if not np.array_equal(merged.number.values,np.arange(11 if hindcast else 51)):
        raise ValueError('Require CF0+PF1..10 reforecast or CF0+PF1..50 realtime')
    if hindcast:
        if set(merged.dims)!={'time','step','number'} or merged.sizes['time']!=20:
            raise ValueError('Require retained 20-year reforecast axes')
        years=pd.DatetimeIndex(merged.time.values)
        if sorted(years.year.tolist())!=list(range(2003,2023)) or years.has_duplicates:
            raise ValueError('Require 2003–2022 exactly once')
    elif set(merged.dims)!={'step','number'}:
        raise ValueError('Unexpected realtime dimensions')
    # Original MLR: concatenate member grids first, then arithmetic spatial mean.
    regional=merged.mean(dim=['latitude','longitude'])
    return regional.load(), audit


def precision_paths(rt, hc):
    """Original native SI order vs exporter PVU-before-float64, no corrections."""
    if rt.dtype != np.dtype('float32') or hc.dtype != np.dtype('float32'):
        raise ValueError('Actual regional SI data are not float32; investigate source dtype, do not cast to force replay')
    native_scalar=hc.mean(dim=list(hc.dims),skipna=True)
    native_anomaly=rt-native_scalar
    native_mean=native_anomaly.mean(dim=[d for d in native_anomaly.dims if d!='number'],skipna=True).sel(number=np.arange(51))
    if native_scalar.dtype != np.dtype('float32') or native_anomaly.dtype != np.dtype('float32') or native_mean.dtype != np.dtype('float32'):
        raise ValueError('Historical intermediate promoted unexpectedly; preserve evidence and investigate')
    old=np.asarray(native_mean.values,dtype=float).squeeze()*1e6
    # Reproduce exporter conversion and its (step,sample) memory/reduction order.
    rp=(rt*1e6).transpose('step','number').values
    hp=(hc*1e6).stack(sample=['time','number']).transpose('step','sample').values
    r64=np.asarray(rp,float);h64=np.asarray(hp,float)
    new=r64.mean(axis=0)-float(h64.mean())
    si64_rt=rt.astype('float64');si64_hc=hc.astype('float64')
    si64_scalar=si64_hc.mean(dim=list(si64_hc.dims),skipna=True)
    si64=(si64_rt-si64_scalar).mean(dim=[d for d in rt.dims if d!='number'],skipna=True).sel(number=np.arange(51)).values*1e6
    native_pvu_scalar=(hc*1e6).mean(dim=list(hc.dims),skipna=True)
    native_pvu=((rt*1e6)-native_pvu_scalar).mean(dim=[d for d in rt.dims if d!='number'],skipna=True).sel(number=np.arange(51)).values.astype(float)
    metadata={'rt_si_dtype':str(rt.dtype),'hc_si_dtype':str(hc.dtype),'rt_dims':list(rt.dims),'hc_dims':list(hc.dims),
              'native_scalar_dtype':str(native_scalar.dtype),'native_scalar_si':float(native_scalar.values),
              'native_subtraction_dtype':str(native_anomaly.dtype),'native_stage_mean_dtype':str(native_mean.dtype),
              'converted_rt_dtype':str(rp.dtype),'converted_hc_dtype':str(hp.dtype),
              'exporter_aggregate_dtype':str(r64.dtype),'exporter_scalar_pvu':float(h64.mean()),
              'si_float64_scalar':float(si64_scalar.values),'native_pvu_scalar':float(native_pvu_scalar.values),
              'rt_step_count':rt.sizes['step'],'hc_step_count':hc.sizes['step'],'hc_samples':hc.sizes['time']*hc.sizes['number']}
    return {'old_mlr_native_si':old,'exporter_pvu_float64':new,'si_float64_then_pvu':si64,'native_pvu_order':native_pvu},metadata

def audit_existing_run(root, manifest):
    original={str(p.relative_to(root)):digest(p) for p in sorted(root.rglob('*')) if p.is_file()}
    for name,expected in manifest['outputs'].items():
        if Path(name).name!=name or original.get(name)!=expected:
            raise ValueError('Existing run artifact hash mismatch')
    qc=pd.read_csv(root/'table_ncvi_daily_stage_replay_qc.csv')
    if len(qc)!=204 or qc.duplicated(['init_date','window','member']).any():
        raise ValueError('Saved replay keys/coverage invalid')
    # Isolate daily-to-new-scalar errors, without changing saved model comparison.
    equal=qc.daily_equal_day_aggregate-qc.mlr_scalar_replay
    weighted=qc.daily_time_weighted_aggregate-qc.mlr_scalar_replay
    if not np.isfinite(np.r_[equal,weighted]).all():raise ValueError('Nonfinite saved internal QC')
    internal={'status':'PASS' if max(equal.abs().max(),weighted.abs().max())<=1e-8 else 'FAIL',
              'max_equal_day_vs_new_scalar':float(equal.abs().max()),
              'max_time_weighted_vs_new_scalar':float(weighted.abs().max()),
              'saved_daily_vs_model_strict_exceedances':int((qc.daily_difference.abs()>1e-8).sum()),
              'existing_formal_process_allowed':manifest['formal_process_allowed']}
    return original,internal

original_hashes, internal_qc = audit_existing_run(RUN_ROOT,run_manifest)
if digest(config['stage_table'])!=run_manifest['inputs']['stage_table']['sha256']:
    raise ValueError('Saved model source changed')
precision_records=[]; precision_intermediates=[]; precision_sources=[]
for precision_init in ('2023-06-08','2023-06-12'):
    r,a=precision_load_si_pair(config['sources'][precision_init]['realtime'],precision_init)
    h,b=precision_load_si_pair(config['sources'][precision_init]['reforecast'],precision_init,True)
    precision_sources.extend(a+b)
    for precision_window,precision_end in [('Stage-I','2023-06-17'),('Total','2023-06-24')]:
        selected=[]
        for arr in [r,h]:
            times=pd.DatetimeIndex(pd.Timestamp(precision_init)+pd.to_timedelta(arr.step.values)+pd.Timedelta(hours=8))
            mask=(times.normalize()>=pd.Timestamp('2023-06-14'))&(times.normalize()<=pd.Timestamp(precision_end))
            if not mask.any():raise ValueError('Missing stage effective times')
            selected.append(arr.isel(step=np.where(mask)[0]))
        paths,metadata=precision_paths(*selected)
        targets=ref[(ref.init_date==precision_init)&(ref.window==precision_window)].set_index('member')
        if len(targets)!=51 or set(targets.index)!=set(range(51)):raise ValueError('Saved target coverage invalid')
        precision_intermediates.append(dict(metadata,init_date=precision_init,window=precision_window))
        for path,values in paths.items():
            for member in range(51):
                saved=float(targets.loc[member,'NCVI_concurrent']);difference=float(values[member]-saved)
                precision_records.append(dict(init_date=precision_init,window=precision_window,member=member,path=path,
                    saved_mlr_pvu=saved,value_pvu=float(values[member]),difference_pvu=difference,
                    strict_1e8_pass=abs(difference)<=1e-8))
precision_table=pd.DataFrame(precision_records)
precision_summary=precision_table.groupby(['init_date','window','path']).agg(
    max_abs_difference=('difference_pvu',lambda x:float(x.abs().max())),
    mean_difference=('difference_pvu','mean'),std_difference_ddof0=('difference_pvu',lambda x:float(x.std(ddof=0))),
    mean_value=('value_pvu','mean'),std_value_ddof0=('value_pvu',lambda x:float(x.std(ddof=0))),
    strict_pass=('strict_1e8_pass','all')).reset_index()
if len(precision_table)!=204*4:raise ValueError('Precision path coverage invalid')
precision_code=get_ipython().history_manager.input_hist_raw[-1]
if not precision_code.strip():raise ValueError('Executed diagnostic code snapshot unavailable')
PRECISION_OUTPUT.mkdir(parents=True,exist_ok=False)
precision_table.to_csv(PRECISION_OUTPUT/'ncvi_precision_member_paths.csv',index=False)
precision_summary.to_csv(PRECISION_OUTPUT/'ncvi_precision_summary.csv',index=False)
pd.DataFrame(precision_intermediates).to_csv(PRECISION_OUTPUT/'ncvi_precision_intermediates.csv',index=False)
pd.DataFrame(precision_sources).to_csv(PRECISION_OUTPUT/'ncvi_precision_sources.csv',index=False)
(PRECISION_OUTPUT/'executed_precision_cell.py').write_text(precision_code,encoding='utf-8')
precision_manifest={'status':'DIAGNOSTIC_ONLY','existing_daily_and_qc_modified':False,'atol':1e-8,
    'reference_commit':AUDIT_REFERENCE_COMMIT,'execution_git_head':None,'runtime':{'python':platform.python_version(),'numpy':np.__version__,'xarray':xr.__version__},
    'model_input_sha256':digest(config['stage_table']),'sources':precision_sources,
    'original_run_sha256':original_hashes,
    'daily_internal_aggregation_qc':internal_qc,
    'historical_mlr_float32_replay_qc':{'status':'PASS' if precision_table[precision_table.path=='old_mlr_native_si'].strict_1e8_pass.all() else 'FAIL',
        'max_abs_difference':float(precision_table[precision_table.path=='old_mlr_native_si'].difference_pvu.abs().max()),
        'comparisons':204,'atol':1e-8},
    'era5_reference_qc':run_manifest['era5_reference_qc'],
    'existing_formal_process_allowed_unchanged':run_manifest['formal_process_allowed'],
    'executed_code_sha256':digest(PRECISION_OUTPUT/'executed_precision_cell.py'),
    'outputs':{p.name:digest(p) for p in PRECISION_OUTPUT.iterdir() if p.is_file()},
    'interpretation':'Native SI replay must reproduce saved MLR separately from daily equal/time-weighted equivalence. Differences are diagnostic, not permission to change thresholds or daily anomalies.'}
if original_hashes!={str(p.relative_to(RUN_ROOT)):digest(p) for p in sorted(RUN_ROOT.rglob('*')) if p.is_file()}:
    raise ValueError('Original run changed during read-only audit')
(PRECISION_OUTPUT/'ncvi_precision_manifest.json').write_text(json.dumps(precision_manifest,indent=2,default=str))
print('Precision diagnostic COMPLETE; existing daily/strict QC unchanged; directory:',PRECISION_OUTPUT)
print(precision_summary.to_string(index=False))
print(pd.DataFrame(precision_intermediates).to_string(index=False))

print(json.dumps({k:precision_manifest[k] for k in ['daily_internal_aggregation_qc','historical_mlr_float32_replay_qc','era5_reference_qc']},indent=2))
